# Information Gain-based Policy Optimization for Multi-Turn Search Agents

Reproduction notebook for Wang et al., ICLR 2026 ([arXiv:2510.14967](https://arxiv.org/abs/2510.14967)).  
Code: https://github.com/ankknaiii/igpo-agentic-search

Outcome-only GRPO collapses when all rollouts in a group share the same reward. IGPO supplies turn-level rewards from adjacent-turn changes in teacher-forced ground-truth probability, then combines them with a final F1 outcome signal.

Runtime: **T4 GPU** recommended. Execute the following cells sequentially.

In [ ]:
#@title Environment setup
import os, sys, subprocess, zipfile
from pathlib import Path

REPO_URL = "https://github.com/ankknaiii/igpo-agentic-search.git"
ROOT = Path("/content/igpo-agentic-search")

def sh(cmd: str):
    print("$", cmd)
    subprocess.check_call(cmd, shell=True)

if not (ROOT / "igpo").exists():
    zip_path = Path("/content/igpo-agentic-search.zip")
    if zip_path.exists():
        with zipfile.ZipFile(zip_path) as z:
            z.extractall("/content")
        if not (ROOT / "igpo").exists() and Path("/content/igpo").exists():
            ROOT = Path("/content")
    else:
        if ROOT.exists():
            sh(f"rm -rf {ROOT}")
        sh(f"git clone --depth 1 {REPO_URL} {ROOT}")

assert (ROOT / "igpo").exists(), f"repository incomplete: {ROOT}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("cwd=", os.getcwd())
sh("pip install -q -r requirements.txt")
sh("pip install -q -e .")

In [ ]:
#@title Unit tests
import subprocess
subprocess.check_call("python scripts/smoke_test.py", shell=True)
subprocess.check_call("pytest -q tests/", shell=True)

In [ ]:
#@title One-step training check
import torch
from igpo.train.trainer import TrainConfig, run_training

print("device=", "cuda:" + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

smoke_cfg = TrainConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    algo="igpo",
    max_steps=1,
    prompts_per_step=1,
    group_size=2,
    max_turns=2,
    max_new_tokens=64,
    learning_rate=1e-5,
    info_gain_type="prob_diff",
    info_gain_norm_mode="separate",
    output_dir="./outputs/smoke",
)
smoke_hist = run_training(smoke_cfg)
smoke_hist[-1]

In [ ]:
#@title IGPO training
from igpo.train.trainer import TrainConfig, run_training

cfg = TrainConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    algo="igpo",  # set "grpo" for the outcome-only baseline
    max_steps=8,
    prompts_per_step=2,
    group_size=4,
    max_turns=3,
    max_new_tokens=128,
    learning_rate=1e-5,
    info_gain_type="prob_diff",
    info_gain_norm_mode="separate",
    output_dir="./outputs/igpo_colab",
)
history = run_training(cfg)
history[-1]

In [ ]:
#@title Training curves
import matplotlib.pyplot as plt

steps = [h.step for h in history]
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].plot(steps, [h.mean_f1 for h in history])
axes[0].set_title("Mean F1")
axes[1].plot(steps, [h.collapse_rate for h in history])
axes[1].set_title("Outcome collapse rate")
axes[2].plot(steps, [h.mean_abs_ig for h in history])
axes[2].set_title("Mean |information gain|")
for ax in axes:
    ax.set_xlabel("step")
plt.tight_layout()
plt.show()

## Notes

- Advantage collapse: identical within-group outcome rewards produce zero z-scored advantages.
- Process reward: $r_t = P(\mathrm{GT}\mid \mathrm{ctx}_t) - P(\mathrm{GT}\mid \mathrm{ctx}_{t-1})$ under teacher forcing.
- Credit assignment: information-gain turns plus final F1, separately normalized, then discounted with $\gamma$.
- Relative to MCTS / external reward models: supervision is intrinsic and ground-truth-aware, with lower annotation cost and reduced reward-hacking surface.